<center><h1>Chen_Yanxiong_HW3</h1></center>
<br>
<br>

Name: Yanxiong Chen
<br>
Github Username: DevinChen02
<br>
USC ID: 2129850673

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
data_dir = Path("../data/AReM")
columns = [
    "time", "avg_rss12", "var_rss12", "avg_rss13",
    "var_rss13", "avg_rss23", "var_rss23"
]

instances = {}
for file_path in sorted(data_dir.glob("*/*.csv")):
    activity = file_path.parent.name
    dataset = int(file_path.stem.replace("dataset", ""))
    instances[(activity, dataset)] = pd.read_csv(
        file_path, comment="#", header=None, names=columns,
        sep=r"[,\s]+", engine="python", usecols=range(7)
    )

data = pd.concat(instances, names=["activity", "dataset", "row"])
row_counts = pd.Series([len(instance) for instance in instances.values()])
data.head()

time  avg_rss12  var_rss12  avg_rss13  var_rss13  \
activity dataset row                                                     
bending1 1       0       0      39.25       0.43      22.75       0.43   
                 1     250      39.25       0.43      23.00       0.00   
                 2     500      39.25       0.43      23.25       0.43   
                 3     750      39.50       0.50      23.00       0.71   
                 4    1000      39.50       0.50      24.00       0.00   

                      avg_rss23  var_rss23  
activity dataset row                        
bending1 1       0        33.75        1.3  
                 1        33.00        0.0  
                 2        33.00        0.0  
                 3        33.00        0.0  
                 4        33.00        0.0

### (b) Test and Train Data

In [3]:
train_instances = {}
test_instances = {}

for (activity, dataset), instance in instances.items():
    test_limit = 2 if activity in ["bending1", "bending2"] else 3
    if dataset <= test_limit:
        test_instances[(activity, dataset)] = instance
    else:
        train_instances[(activity, dataset)] = instance

print("Training instances:", len(train_instances))
print("Test instances:", len(test_instances))

Training instances: 69
Test instances: 19


### (c) Feature Extraction

#### i. Research

Types of time-domain features that are usually used in time series classification:

Minimum, maximum, mean, median, standard deviation, variance, first quartile, third quartile, root mean square, absolute energy, skewness, kurtosis, autocorrelation, linear trend, sum of absolute consecutive changes, peak count, and threshold-crossing count.

Sources: https://tsfresh.readthedocs.io/en/stable/text/list_of_features.html; 
https://tsfresh.readthedocs.io/en/stable/api/tsfresh.feature_extraction.html#tsfresh.feature_extraction.feature_calculators.quantile

#### ii. Extraction

I did not normalize or standardize the data. I calculated the features from the original signal values. Standard deviation uses the sample convention, with an n - 1 denominator for variance.

Lecture references: Lesson 3, *Classification*, PDF p. 6 (quartiles) and p. 121 (sample variance).

In [4]:
statistic_names = {
    "min": "min", "max": "max", "mean": "mean", "50%": "median",
    "std": "std", "25%": "q1", "75%": "q3"
}

feature_rows = []
for instance in instances.values():
    summary = instance[columns[1:]].describe()
    feature_rows.append({
        f"{name}_{i}": summary.loc[statistic, signal]
        for i, signal in enumerate(columns[1:], 1)
        for statistic, name in statistic_names.items()
    })

features = pd.DataFrame(
    feature_rows,
    index=pd.MultiIndex.from_tuples(instances.keys(), names=["activity", "dataset"])
)

print("Feature table shape:", features.shape)
features.head()

Feature table shape: (88, 42)


min_1  max_1     mean_1  median_1     std_1   q1_1   q3_1  \
activity dataset                                                              
bending1 1        37.25  45.00  40.624792     40.50  1.476967  39.25  42.00   
         2        38.00  45.67  42.812812     42.50  1.435550  42.00  43.67   
         3        35.00  47.40  43.954500     44.33  1.558835  43.00  45.00   
         4        33.00  47.75  42.179812     43.50  3.670666  39.15  45.00   
         5        33.00  45.75  41.678063     41.75  2.243490  41.33  42.75   

                  min_2  max_2    mean_2  ...     std_5     q1_5   q3_5  \
activity dataset                          ...                             
bending1 1          0.0   1.30  0.358604  ...  2.188449  33.0000  36.00   
         2          0.0   1.22  0.372437  ...  1.995255  32.0000  34.50   
         3          0.0   1.70  0.426250  ...  1.999604  35.3625  36.50   
         4          0.0   3.00  0.696042  ...  3.849448  30.4575  36.33   
         5          0.0   2.83  0.535979  ...  2.411026  28.4575  31.25   

                  min_6  max_6    mean_6  median_6     std_6  q1_6  q3_6  
activity dataset                                                          
bending1 1          0.0   1.92  0.570583      0.43  0.582915   0.0  1.30  
         2          0.0   3.11  0.571083      0.43  0.601010   0.0  1.30  
         3          0.0   1.79  0.493292      0.43  0.513506   0.0  0.94  
         4          0.0   2.18  0.613521      0.50  0.524317   0.0  1.00  
         5          0.0   1.79  0.383292      0.43  0.389164   0.0  0.50  

[5 rows x 42 columns]

#### iii. Standard Deviation

Reference: Lesson 4, *Resampling*, PDF pp. 65-67 (bootstrap percentile confidence intervals).

In [5]:
rng = np.random.default_rng(42)
n_bootstrap = 10000
values = features.to_numpy()
bootstrap_sd = np.empty((n_bootstrap, features.shape[1]))

for b in range(n_bootstrap):
    sample = values[rng.integers(0, len(values), size=len(values))]
    bootstrap_sd[b] = sample.std(axis=0, ddof=1)

lower, upper = np.percentile(bootstrap_sd, [5, 95], axis=0)
feature_uncertainty = pd.DataFrame({
    "standard_deviation": features.std(ddof=1),
    "90% CI lower": lower,
    "90% CI upper": upper
})
feature_uncertainty.round(4)

,standard_deviation,90% CI lower,90% CI upper
min_1,9.5700,8.2591,10.7762
max_1,4.3944,3.3243,5.2928
mean_1,5.3357,4.6952,5.8836
median_1,5.4401,4.7909,6.0076
std_1,1.7722,1.5701,1.9549
q1_1,6.1536,5.5646,6.6560
q3_1,5.1389,4.3111,5.8451
min_2,0.0000,0.0000,0.0000
max_2,5.0627,4.6254,5.4020
mean_2,1.5742,1.3977,1.7068


#### iv. Select Features

I choose mean, standard deviation, and third quartile (Q3) as the three time-domain feature types. Each is calculated for all six signals.

The mean describes the typical signal level. Standard deviation measures how much the signal varies within an instance. Q3 describes the upper part of the signal's values and is less sensitive to a single extreme value than the maximum.

Part (iii) shows that all six extracted columns for each of these feature types vary across the 88 instances. For example:

| Extracted feature | Standard deviation across instances | 90% bootstrap confidence interval |
| --- | --- | --- |
| mean_1 | 5.3357 | [4.6952, 5.8836] |
| std_1 | 1.7722 | [1.5701, 1.9549] |
| q3_1 | 5.1389 | [4.3111, 5.8451] |

I would leave out minimum because min_2 and min_4 are constant. Their standard deviations are zero, with confidence intervals of [0, 0]. These two columns cannot distinguish instances.

My choice uses the variation shown in part (iii) and what each feature measures. A larger standard deviation does not automatically make a feature more important. A narrower confidence interval means the standard deviation is estimated more precisely.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model will have a lower or equal training RSS. Setting $\beta_2 = \beta_3 = 0$ gives the linear model, so the cubic model can always match its fit. Least squares chooses the coefficients that minimize training RSS. The extra terms therefore cannot make the training fit worse, even when the true relationship is linear.

### (b) Linear Test

We expect the linear model to have a lower test RSS. It already matches the true relationship. The cubic model adds unnecessary terms that can fit noise in the training data. This increases variance without reducing bias. The result can differ on a particular test set, but the linear model should perform better on average.

### (c) Not Linear Train

The cubic model will still have a lower or equal training RSS. It includes the linear model as a special case, so its extra terms cannot increase the minimum training RSS. This holds even when the true relationship is nonlinear.

### (d) Not Linear Testing

There is not enough information to tell which model will have a lower test RSS. If the relationship is close to linear, the linear model may do better because it has lower variance. If the cubic terms capture enough of the curvature, the cubic model may do better by reducing bias. We do not know whether that improvement is large enough to outweigh the extra variance.